<a href="https://colab.research.google.com/github/talpt/pyton/blob/main/Merdiven_Hacim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# MEERDİVEN HACİM TARAMASI, YEŞİL BAR ŞARTI EKLENDİ...17.08.2025...

# Gerekli kütüphaneleri yükle (sadece ilk çalıştırmada)
!pip install pandas numpy tqdm openpyxl
!pip install git+https://github.com/rongardF/tvdatafeed
!pip install tradingview-screener==2.5.0

import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
from tvDatafeed import Interval
from tradingview_screener import get_all_symbols
from tvDatafeed import TvDatafeed
from IPython.display import display

# TvDatafeed bağlantısı
# Not: Bazen ilk bağlantıda hata olabilir, tekrar denemek gerekebilir.
try:
    tv = TvDatafeed()
except Exception as e:
    print(f"TvDatafeed bağlantı hatası: {e}. Tekrar deneniyor...")
    try:
        tv = TvDatafeed()
    except Exception as e2:
        print(f"TvDatafeed ikinci bağlantı hatası: {e2}. Program sonlandırılıyor.")
        exit()

# ✅ Zaman periyodu seçim fonksiyonu
def zaman_periyodu_seçimi():
    print("\n📆 Tarama Zaman Periyodunu Seçiniz:")
    print("1 - Günlük\n2 - Haftalık\n3 - Aylık\n4 - 1 Saat\n5 - 4 Saat")
    secim = input("Seçim (1/2/3/4/5): ").strip()

    interval_map = {
        "1": (Interval.in_daily, "Günlük"),
        "2": (Interval.in_weekly, "Haftalık"),
        "3": (Interval.in_monthly, "Aylık"),
        "4": (Interval.in_1_hour, "1 Saatlik"),
        "5": (Interval.in_4_hour, "4 Saatlik")
    }

    # Geçersiz seçimde varsayılan olarak Günlük
    interval, interval_label = interval_map.get(secim, (Interval.in_daily, "Günlük"))
    print(f"Seçilen zaman periyodu: {interval_label}")
    return interval, interval_label

# ✅ Görünüm ayarı
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.colheader_justify', 'left')

# ✅ Stil fonksiyonu (Yeni sütunlara göre güncellendi)
def format_tarama_sonucu(df):
    # Sayısal sütunları belirle
    # Hacim genellikle tam sayı veya çok az ondalıklı olabilir. SMA ondalıklı olur.
    numerical_cols = [col for col in df.columns if col in ['Son Hacim', 'Önceki Hacim', '2 Önceki Hacim', '3 Önceki Hacim', f'Hacim SMA({10})']]

    text_cols = ['Hisse']

    # Sayısal sütunlar için format
    format_dict = {}
    for col in numerical_cols:
        if 'SMA' in col:
             format_dict[col] = '{:,.2f}' # SMA için 2 ondalık
        else:
             format_dict[col] = '{:,.0f}' # Hacim için tam sayı (veya isteğe bağlı olarak ondalık)


    # Hizalama ayarları
    text_align_props = {'text-align': 'left', 'font-size': '12px'}
    center_align_props = {'text-align': 'center', 'font-size': '12px'}

    # Başlık stili
    header_props = [('background-color', '#2A5CAA'), ('color', 'white'), ('text-align', 'center')]

    # Stili uygula
    styled_df = (df.style
        .set_properties(**text_align_props, subset=text_cols)
        .set_properties(**center_align_props, subset=numerical_cols)
        .set_table_styles([{
            'selector': 'th',
            'props': header_props
        }])
        .background_gradient(cmap='Blues', subset=numerical_cols)
        .format(format_dict))

    return styled_df

# Ana tarama fonksiyonu
def main():
    interval, interval_label = zaman_periyodu_seçimi()

    hisseler = sorted([s.replace('BIST:', '') for s in get_all_symbols(market='turkey')])
    sonuclar = []

    # ✅ Strateji parametreleri
    sma_period = 10 # Hacim SMA periyodu

    # Koşullar için gereken minimum çubuk sayısı
    # SMA(10) için 10 çubuk, ref(v, -3) için 4 çubuk. En az 10 çubuk lazım.
    min_bars_required = sma_period

    # Güvenlik için biraz daha fazla veri çekelim
    n_bars_to_fetch = max(50, min_bars_required + 5) # En az 50 veya gereken + 5

    for hisse in tqdm(hisseler, desc="Tarama Yapılıyor", unit="hisse"):
        try:
            # Yeterli veri çekmeye çalış
            data = tv.get_hist(hisse, exchange='BIST', interval=interval, n_bars=n_bars_to_fetch)

            # Veri kontrolü
            # SMA hesaplandıktan ve NaN'lar temizlendikten sonra en az 4 çubuk kalmalı
            if data is None or data.empty or len(data) < min_bars_required:
                # print(f"'{hisse}' için yeterli veri yok ({len(data)} çubuk). Atlanıyor.")
                continue

            # ✅ Hacim SMA hesapla
            data['volume_sma10'] = data['volume'].rolling(window=sma_period).mean()

            # Hesaplamalardan kaynaklanan ilk NaN değerleri temizle
            data = data.dropna(subset=['volume', 'volume_sma10'])

            if len(data) < 4:
                continue

            # ✅ "Yeşil bar" tanımı
            # Tercih 1 (varsayılan): Kapanış > Açılış
            data['green_co'] = data['close'] > data['open']
            # Tercih 2 (opsiyonel): Kapanış > Önceki Kapanış
            data['green_pc'] = data['close'] > data['close'].shift(1)

            # Hangi tanımı kullanacağını buradan seç:
            yesil_tanim = 'co'  # 'co' veya 'pc'
            green_col = 'green_co' if yesil_tanim == 'co' else 'green_pc'

            # ✅ Son çubuklar için değerler
            last_volume  = data['volume'].iloc[-1]
            prev_volume  = data['volume'].iloc[-2]
            prev2_volume = data['volume'].iloc[-3]
            prev3_volume = data['volume'].iloc[-4]
            last_volume_sma = data['volume_sma10'].iloc[-1]

            # ✅ Hacim artışı (merdiven)
            condition_increasing_volume = (last_volume > prev_volume) and \
                                          (prev_volume > prev2_volume) and \
                                          (prev2_volume > prev3_volume)

            # ✅ SMA üstü
            condition_above_sma = last_volume > last_volume_sma

            # ✅ Yeşil bar şartı
            # Sıkı mod: Son 3 bar da yeşil olsun (hacmin arttığı barlar)
            condition_green_bars_strict = bool(data[green_col].iloc[-1] and
                                               data[green_col].iloc[-2] and
                                               data[green_col].iloc[-3])

            # Esnek mod: Sadece son bar yeşil olsun
            condition_green_bars_flexible = bool(data[green_col].iloc[-1])

            # Hangi modu istediğini buradan seç
            yesil_mod = 'strict'  # 'strict' veya 'flex'
            condition_green = condition_green_bars_strict if yesil_mod == 'strict' else condition_green_bars_flexible

            # ✅ Tüm koşullar
            if condition_increasing_volume and condition_above_sma and condition_green:
                hisse_sonuc = {
                    'Hisse': hisse,
                    'Son Hacim': last_volume,
                    'Önceki Hacim': prev_volume,
                    '2 Önceki Hacim': prev2_volume,
                    '3 Önceki Hacim': prev3_volume,
                    f'Hacim SMA({sma_period})': last_volume_sma,
                    'Son Bar Yeşil mi?': 'Evet' if data[green_col].iloc[-1] else 'Hayır'
                }
                sonuclar.append(hisse_sonuc)

        except Exception as e:
            # Hata mesajlarını yorum satırı yaparak çıktıyı temiz tutabilirsiniz.
            # import traceback
            # print(f"'{hisse}' için hata: {e}")
            # traceback.print_exc()
            pass # Hata durumunda hisseyi atla ve devam et

    if sonuclar:
        df = pd.DataFrame(sonuclar)

        # ✅ Sonuç sütunlarının sırası
        ordered_cols = [
            'Hisse',
            'Son Hacim',
            'Önceki Hacim',
            '2 Önceki Hacim',
            '3 Önceki Hacim',
            f'Hacim SMA({sma_period})',
            'Son Bar Yeşil mi?'
        ]

        # DataFrame'deki mevcut sütunları kullanarak sıralama yap
        final_cols = [col for col in ordered_cols if col in df.columns]
        df = df[final_cols]

        print(f"\n🎯 {len(df)} hisse bulundu.")
        # ✅ Stil fonksiyonu ile tabloyu göster
        display(format_tarama_sonucu(df))

        # ✅ BİLGİLENDİRME MESAJI (Bu projeye özel)
        tarih_saat = datetime.now().strftime("%Y-%m-%d %H:%M")

        print(f"""\n📋 {tarih_saat} tarihinde, belirlenen 'Artan Hacim ve Ortalamanın Üzeri' stratejisi koşulları doğrultusunda:
📊 Zaman Aralığı: {interval_label}
⏳ Hacim SMA Periyodu: {sma_period}
🔍 Strateji Koşulları:
    1. Son 3 periyottur YEŞİL (bar) hacim artışı gözlemlenmeli (Bugünkü Hacim > Önceki Hacim > 2 Önceki Hacim > 3 Önceki Hacim).
    2. Bugünkü hacim, son {sma_period} periyotluk basit hareketli ortalamanın üzerinde olmalı. YEŞİL BAR ŞARTI EKLENDİ...!!!!
Koşullarına göre toplam {len(df)} adet hisse bulunmuştur.
""")

        # Dosya adına strateji bilgisini ekle
        filename = f"BIST_Artan_Hacim_TARAMA_{interval_label}_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
        try:
            # ✅ DataFrame'i kaydet
            df.to_excel(filename, index=False)
            print(f"\n💾 Excel'e kaydedildi: {filename}")
        except Exception as e:
            print(f"\n❌ Excel'e kaydetme hatası: {e}")

    else:
        print("\n⚠️ Uygun hisse bulunamadı veya veri çekilemedi.")

# Çalıştır
if __name__ == "__main__":
    main()


  Cloning https://github.com/rongardF/tvdatafeed to /tmp/pip-req-build-oiagn46s
  Running command git clone --filter=blob:none --quiet https://github.com/rongardF/tvdatafeed /tmp/pip-req-build-oiagn46s
  Resolved https://github.com/rongardF/tvdatafeed to commit e6f6aaa7de439ac6e454d9b26d2760ded8dc4923
  Preparing metadata (setup.py) ... done



📆 Tarama Zaman Periyodunu Seçiniz:
1 - Günlük
2 - Haftalık
3 - Aylık
4 - 1 Saat
5 - 4 Saat
Seçim (1/2/3/4/5): 1
Seçilen zaman periyodu: Günlük


Tarama Yapılıyor: 100%|██████████| 607/607 [05:10<00:00,  1.95hisse/s]


🎯 8 hisse bulundu.


,Hisse,Son Hacim,Önceki Hacim,2 Önceki Hacim,3 Önceki Hacim,Hacim SMA(10),Son Bar Yeşil mi?
0,ALKIM,"2,940,319","1,844,708","1,512,753","1,282,939","2,203,351.70",Evet
1,BAHKM,"3,230,440","1,736,324","1,589,786","1,078,020","1,521,866.50",Evet
2,BIGCH,"5,005,777","4,761,933","2,305,290","1,991,280","2,719,346.90",Evet
3,DOHOL,"33,000,987","32,411,381","25,420,646","18,328,512","21,135,758.20",Evet
4,GLBMD,"2,256,531","2,079,315","1,819,269","604,060","1,152,637.20",Evet
5,HEKTS,"638,396,160","352,996,310","315,464,730","207,110,300","282,837,205.20",Evet
6,PETKM,"64,491,072","56,594,302","34,071,400","25,426,459","48,445,249.30",Evet
7,SASA,"1,492,223,570","1,141,676,250","1,136,743,520","537,298,920","843,030,238.00",Evet



📋 2025-08-17 14:59 tarihinde, belirlenen 'Artan Hacim ve Ortalamanın Üzeri' stratejisi koşulları doğrultusunda:
📊 Zaman Aralığı: Günlük
⏳ Hacim SMA Periyodu: 10
🔍 Strateji Koşulları:
    1. Son 3 periyottur YEŞİL (bar) hacim artışı gözlemlenmeli (Bugünkü Hacim > Önceki Hacim > 2 Önceki Hacim > 3 Önceki Hacim).
    2. Bugünkü hacim, son 10 periyotluk basit hareketli ortalamanın üzerinde olmalı. YEŞİL BAR ŞARTI EKLENDİ...!!!!
Koşullarına göre toplam 8 adet hisse bulunmuştur.


💾 Excel'e kaydedildi: BIST_Artan_Hacim_TARAMA_Günlük_20250817_1459.xlsx


In [ ]:
#📌 Ne Anlama Geliyor?
#Bu koşul hacim (volume) için yazılmıştır. Şartlar:

# YEŞİL HACİM BAR' I ŞARTI EKLENDİ...!!!!!

#v > ref(v, -1)
#→ Bugünkü hacim, dünkü hacimden büyük.

#ref(v, -1) > ref(v, -2)
#→ Dünkü hacim, 2 gün önceki hacimden büyük.

#ref(v, -2) > ref(v, -3)
#→ 2 gün önceki hacim, 3 gün öncekinden büyük.

#v > mov(vol(), 10, S)
#→ Bugünkü hacim, son 10 günlük basit hacim ortalamasının (SMA) üzerinde.

#✅ Yani bu formül şunu arar:
#Üst üste 3 gündür artan bir hacim trendi olacak

#Bugünkü hacim ayrıca son 10 günlük ortalamanın da üzerinde olacak

#📈 Yatırımcı için anlamı:
#Bu formül, hacimle güçlenen ilgiyi tespit eder. Özellikle:

#Yeni başlayan yükselişlerde

#Breakout (direnç kırılımı) öncesinde

#Manipülasyon veya haber etkisiyle oluşabilecek ani ilgi dalgalarında kullanışlıdır.

#🎯 Opsiyonel Geliştirme:
#Bu formül hacim bazlı. İstenirse şunlar da dahil edilebilir:

#Fiyat > EMA veya kapanış > direnç seviyeleri

#RSI > 50 gibi momentum kriterleri

#Son fiyatın % olarak ortalamanın ne kadar üzerinde olduğu

In [ ]:
# MERDİVEN HACİM TARAMASI, YEŞİL BAR ŞARTI EKLENDİ...17.08.2025...

# Gerekli kütüphaneleri yükle (sadece ilk çalıştırmada)
!pip install pandas numpy tqdm openpyxl
!pip install git+https://github.com/rongardF/tvdatafeed
!pip install tradingview-screener==2.5.0

import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
from tvDatafeed import Interval
from tradingview_screener import get_all_symbols
from tvDatafeed import TvDatafeed
from IPython.display import display

# TvDatafeed bağlantısı
# Not: Bazen ilk bağlantıda hata olabilir, tekrar denemek gerekebilir.
try:
    tv = TvDatafeed()
except Exception as e:
    print(f"TvDatafeed bağlantı hatası: {e}. Tekrar deneniyor...")
    try:
        tv = TvDatafeed()
    except Exception as e2:
        print(f"TvDatafeed ikinci bağlantı hatası: {e2}. Program sonlandırılıyor.")
        exit()

# ✅ Zaman periyodu seçim fonksiyonu
def zaman_periyodu_seçimi():
    print("\n📆 Tarama Zaman Periyodunu Seçiniz:")
    print("1 - Günlük\n2 - Haftalık\n3 - Aylık\n4 - 1 Saat\n5 - 4 Saat")
    secim = input("Seçim (1/2/3/4/5): ").strip()

    interval_map = {
        "1": (Interval.in_daily, "Günlük"),
        "2": (Interval.in_weekly, "Haftalık"),
        "3": (Interval.in_monthly, "Aylık"),
        "4": (Interval.in_1_hour, "1 Saatlik"),
        "5": (Interval.in_4_hour, "4 Saatlik")
    }

    # Geçersiz seçimde varsayılan olarak Günlük
    interval, interval_label = interval_map.get(secim, (Interval.in_daily, "Günlük"))
    print(f"Seçilen zaman periyodu: {interval_label}")
    return interval, interval_label

# ✅ Görünüm ayarı
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.colheader_justify', 'left')

# ✅ Stil fonksiyonu (sma_period parametreli)
def format_tarama_sonucu(df, sma_period=10):
    # Sayısal sütunları belirle
    numerical_cols = [col for col in df.columns if col in [
        'Son Hacim', 'Önceki Hacim', '2 Önceki Hacim', '3 Önceki Hacim', f'Hacim SMA({sma_period})'
    ]]
    text_cols = ['Hisse']

    # Sayısal sütunlar için format
    format_dict = {}
    for col in numerical_cols:
        if 'SMA' in col:
            format_dict[col] = '{:,.2f}'  # SMA için 2 ondalık
        else:
            format_dict[col] = '{:,.0f}'  # Hacim için tam sayı

    # Hizalama ayarları
    text_align_props = {'text-align': 'left', 'font-size': '12px'}
    center_align_props = {'text-align': 'center', 'font-size': '12px'}

    # Başlık stili
    header_props = [('background-color', '#2A5CAA'), ('color', 'white'), ('text-align', 'center')]

    # Stili uygula
    styled_df = (df.style
        .set_properties(**text_align_props, subset=text_cols)
        .set_properties(**center_align_props, subset=numerical_cols)
        .set_table_styles([{
            'selector': 'th',
            'props': header_props
        }])
        .background_gradient(cmap='Blues', subset=numerical_cols)
        .format(format_dict))

    return styled_df

# Ana tarama fonksiyonu (tam script)
def main():
    interval, interval_label = zaman_periyodu_seçimi()

    hisseler = sorted([s.replace('BIST:', '') for s in get_all_symbols(market='turkey')])
    sonuclar_full = []     # Merdiven + Yeşil + SMA üstü
    sonuclar_ladder = []   # Merdiven + Yeşil (SMA yok)

    # ✅ Strateji parametreleri
    sma_period = 10  # Hacim SMA periyodu

    # Koşullar için gereken minimum çubuk sayısı
    min_bars_required = sma_period

    # Güvenlik için biraz daha fazla veri çekelim
    n_bars_to_fetch = max(50, min_bars_required + 5)

    # Kullanıcı ayarları (döngü dışında, kolayca değiştirebilirsin)
    yesil_tanim = 'co'   # 'co' (kapanış>açılış) veya 'pc' (kapanış>önceki kapanış)
    yesil_mod = 'strict' # 'strict' (son 3 bar da yeşil) veya 'flex' (sadece son bar yeşil)

    for hisse in tqdm(hisseler, desc="Tarama Yapılıyor", unit="hisse"):
        try:
            # Veri çek
            data = tv.get_hist(hisse, exchange='BIST', interval=interval, n_bars=n_bars_to_fetch)

            # Veri kontrolü
            if data is None or data.empty or len(data) < min_bars_required:
                continue

            # Hacim SMA
            data['volume_sma10'] = data['volume'].rolling(window=sma_period).mean()

            # NaN temizle
            data = data.dropna(subset=['volume', 'volume_sma10'])
            if len(data) < 4:
                continue

            # Yeşil bar tanımları
            data['green_co'] = data['close'] > data['open']
            data['green_pc'] = data['close'] > data['close'].shift(1)
            green_col = 'green_co' if yesil_tanim == 'co' else 'green_pc'

            # Son çubuk değerleri
            last_volume  = data['volume'].iloc[-1]
            prev_volume  = data['volume'].iloc[-2]
            prev2_volume = data['volume'].iloc[-3]
            prev3_volume = data['volume'].iloc[-4]
            last_volume_sma = data['volume_sma10'].iloc[-1]

            # Hacim merdiven koşulu
            condition_increasing_volume = (last_volume > prev_volume) and \
                                          (prev_volume > prev2_volume) and \
                                          (prev2_volume > prev3_volume)

            # SMA üstü kontrol
            condition_above_sma = last_volume > last_volume_sma

            # Yeşil bar şartları
            condition_green_bars_strict = bool(data[green_col].iloc[-1] and
                                               data[green_col].iloc[-2] and
                                               data[green_col].iloc[-3])
            condition_green_bars_flexible = bool(data[green_col].iloc[-1])
            condition_green = condition_green_bars_strict if yesil_mod == 'strict' else condition_green_bars_flexible

            # Merdiven + Yeşil (SMA yok) kaydı
            if condition_increasing_volume and condition_green:
                base_rec = {
                    'Hisse': hisse,
                    'Son Hacim': last_volume,
                    'Önceki Hacim': prev_volume,
                    '2 Önceki Hacim': prev2_volume,
                    '3 Önceki Hacim': prev3_volume,
                    f'Hacim SMA({sma_period})': last_volume_sma,
                    'Son Bar Yeşil mi?': 'Evet' if data[green_col].iloc[-1] else 'Hayır'
                }
                sonuclar_ladder.append(base_rec)

                # Eğer SMA üstü de sağlanıyorsa "full" listesine ekle
                if condition_above_sma:
                    sonuclar_full.append(base_rec.copy())

        except Exception as e:
            # Hata durumunda hisseyi atla (debug için print açabilirsiniz)
            # print(f"'{hisse}' için hata: {e}")
            pass

    # Döngü sonrası: sonuçları düzenle, karşılaştır ve göster
    ordered_cols = [
        'Hisse',
        'Son Hacim',
        'Önceki Hacim',
        '2 Önceki Hacim',
        '3 Önceki Hacim',
        f'Hacim SMA({sma_period})',
        'Son Bar Yeşil mi?'
    ]

    df_ladder = pd.DataFrame(sonuclar_ladder) if sonuclar_ladder else pd.DataFrame(columns=ordered_cols)
    df_full   = pd.DataFrame(sonuclar_full)   if sonuclar_full   else pd.DataFrame(columns=ordered_cols)

    df_ladder = df_ladder[[c for c in ordered_cols if c in df_ladder.columns]]
    df_full   = df_full[[c for c in ordered_cols if c in df_full.columns]]

    # "Ladder yalnızca" (SMA şartını sağlamayanlar) — Full listesindeki hisseleri çıkar
    df_ladder_only = df_ladder[~df_ladder['Hisse'].isin(df_full['Hisse'])].reset_index(drop=True)

    # Sayılar
    n_ladder_all  = len(df_ladder)
    n_ladder_only = len(df_ladder_only)
    n_full        = len(df_full)

    print(f"\n🎯 MERDİVEN + YEŞİL (TÜM): {n_ladder_all} hisse")
    if n_ladder_all:
        display(format_tarama_sonucu(df_ladder, sma_period))

    print(f"\n🔎 MERDİVEN + YEŞİL (Sadece SMA ŞARTSIZ - Ladder Only): {n_ladder_only} hisse")
    if n_ladder_only:
        display(format_tarama_sonucu(df_ladder_only, sma_period))

    print(f"\n✅ MERDİVEN + YEŞİL + SMA({sma_period}) ÜSTÜ (Full): {n_full} hisse")
    if n_full:
        display(format_tarama_sonucu(df_full, sma_period))

    # Hızlı debug: Ladder'da olup Full'de olmayan ilk birkaç hisseyi yazdır (varsa)
    if n_ladder_only:
        print("\nÖrnek - Ladder Only (ilk 10):")
        print(df_ladder_only[['Hisse', 'Son Hacim', f'Hacim SMA({sma_period})']].head(10).to_string(index=False))

    # Bilgilendirme metni
    tarih_saat = datetime.now().strftime("%Y-%m-%d %H:%M")
    print(f"""\n📋 {tarih_saat} — Tarama Özeti
📊 Zaman Aralığı: {interval_label}
⏳ Hacim SMA Periyodu: {sma_period}
🔍 Koşullar:
  • Liste (Tüm Ladder): Son 3 bar hacim artışı + Yeşil bar şartı (SMA kontrolü gösteriliyor ama zorunlu değil)
  • Ladder Only: Ladder içinde olup SMA üstü olmayanlar
  • Full: Ladder + Son bar hacmi SMA({sma_period}) üstü
Toplam: {n_ladder_all} (ladder tüm) / {n_ladder_only} (ladder only) / {n_full} (full)
""")

    # Excel'e yaz: 3 sayfa (Ladder_All, Ladder_Only_noSMA, Full)
    filename = f"BIST_Artan_Hacim_TARAMA_{interval_label}_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df_ladder.to_excel(writer, index=False, sheet_name='Ladder_All')
            df_ladder_only.to_excel(writer, index=False, sheet_name='Ladder_Only_noSMA')
            df_full.to_excel(writer, index=False, sheet_name='Ladder_plus_SMA')
        print(f"\n💾 Excel'e kaydedildi: {filename} (3 sayfa)")
    except Exception as e:
        print(f"\n❌ Excel'e kaydetme hatası: {e}")

# Çalıştır
if __name__ == "__main__":
    main()


  Cloning https://github.com/rongardF/tvdatafeed to /tmp/pip-req-build-27tgt7hq
  Running command git clone --filter=blob:none --quiet https://github.com/rongardF/tvdatafeed /tmp/pip-req-build-27tgt7hq
  Resolved https://github.com/rongardF/tvdatafeed to commit e6f6aaa7de439ac6e454d9b26d2760ded8dc4923
  Preparing metadata (setup.py) ... done



📆 Tarama Zaman Periyodunu Seçiniz:
1 - Günlük
2 - Haftalık
3 - Aylık
4 - 1 Saat
5 - 4 Saat
Seçim (1/2/3/4/5): 1
Seçilen zaman periyodu: Günlük


Tarama Yapılıyor: 100%|██████████| 607/607 [05:05<00:00,  1.99hisse/s]


🎯 MERDİVEN + YEŞİL (TÜM): 9 hisse


,Hisse,Son Hacim,Önceki Hacim,2 Önceki Hacim,3 Önceki Hacim,Hacim SMA(10),Son Bar Yeşil mi?
0,ALKIM,"2,940,319","1,844,708","1,512,753","1,282,939","2,203,351.70",Evet
1,BAHKM,"3,230,440","1,736,324","1,589,786","1,078,020","1,521,866.50",Evet
2,BIGCH,"5,005,777","4,761,933","2,305,290","1,991,280","2,719,346.90",Evet
3,DOAS,"1,946,773","1,855,987","1,448,951","1,206,643","2,499,479.10",Evet
4,DOHOL,"33,000,987","32,411,381","25,420,646","18,328,512","21,135,758.20",Evet
5,GLBMD,"2,256,531","2,079,315","1,819,269","604,060","1,152,637.20",Evet
6,HEKTS,"638,396,160","352,996,310","315,464,730","207,110,300","282,837,205.20",Evet
7,PETKM,"64,491,072","56,594,302","34,071,400","25,426,459","48,445,249.30",Evet
8,SASA,"1,492,223,570","1,141,676,250","1,136,743,520","537,298,920","843,030,238.00",Evet



🔎 MERDİVEN + YEŞİL (Sadece SMA ŞARTSIZ - Ladder Only): 1 hisse


,Hisse,Son Hacim,Önceki Hacim,2 Önceki Hacim,3 Önceki Hacim,Hacim SMA(10),Son Bar Yeşil mi?
0,DOAS,"1,946,773","1,855,987","1,448,951","1,206,643","2,499,479.10",Evet



✅ MERDİVEN + YEŞİL + SMA(10) ÜSTÜ (Full): 8 hisse


,Hisse,Son Hacim,Önceki Hacim,2 Önceki Hacim,3 Önceki Hacim,Hacim SMA(10),Son Bar Yeşil mi?
0,ALKIM,"2,940,319","1,844,708","1,512,753","1,282,939","2,203,351.70",Evet
1,BAHKM,"3,230,440","1,736,324","1,589,786","1,078,020","1,521,866.50",Evet
2,BIGCH,"5,005,777","4,761,933","2,305,290","1,991,280","2,719,346.90",Evet
3,DOHOL,"33,000,987","32,411,381","25,420,646","18,328,512","21,135,758.20",Evet
4,GLBMD,"2,256,531","2,079,315","1,819,269","604,060","1,152,637.20",Evet
5,HEKTS,"638,396,160","352,996,310","315,464,730","207,110,300","282,837,205.20",Evet
6,PETKM,"64,491,072","56,594,302","34,071,400","25,426,459","48,445,249.30",Evet
7,SASA,"1,492,223,570","1,141,676,250","1,136,743,520","537,298,920","843,030,238.00",Evet



Örnek - Ladder Only (ilk 10):
Hisse  Son Hacim  Hacim SMA(10)
DOAS  1946773.0  2499479.1     

📋 2025-08-17 15:06 — Tarama Özeti
📊 Zaman Aralığı: Günlük
⏳ Hacim SMA Periyodu: 10
🔍 Koşullar:
  • Liste (Tüm Ladder): Son 3 bar hacim artışı + Yeşil bar şartı (SMA kontrolü gösteriliyor ama zorunlu değil)
  • Ladder Only: Ladder içinde olup SMA üstü olmayanlar
  • Full: Ladder + Son bar hacmi SMA(10) üstü
Toplam: 9 (ladder tüm) / 1 (ladder only) / 8 (full)


💾 Excel'e kaydedildi: BIST_Artan_Hacim_TARAMA_Günlük_20250817_1506.xlsx (3 sayfa)
